# Task 3: Local 7B Parameter LLM Quantization & Logit Extraction Pipeline

## Mathematics
- 4-bit Uniform Quantization: Scale $S = \frac{x_{max} - x_{min}}{2^b - 1}$, Zero Point $Z = \text{round}(\frac{-x_{min}}{S})$.
- Dynamic Shannon Entropy: $H(X) = - \sum P(x) \log_2 P(x)$.


In [ ]:
import torch
import torch.nn.functional as F

# 4-bit integer quantization and dynamic logit entropy computation functions
def quantize_int4(tensor: torch.Tensor):
    qmin, qmax = 0, 15
    min_val, max_val = tensor.min(), tensor.max()
    scale = (max_val - min_val) / (qmax - qmin)
    zero_point = torch.round(-min_val / scale)
    q_tensor = torch.clamp(torch.round(tensor / scale) + zero_point, qmin, qmax).to(torch.uint8)
    return q_tensor, scale, zero_point

def compute_entropy(logits: torch.Tensor):
    probs = F.softmax(logits, dim=-1)
    return -torch.sum(probs * torch.log2(probs + 1e-9), dim=-1)


In [ ]:
# Test 4-bit quantization compression efficiency and compute step logit entropy
weights = torch.randn(500, 500)
q_w, s, zp = quantize_int4(weights)
recon = s * (q_w.to(torch.float32) - zp)

mse = torch.mean((weights - recon)**2).item()
entropy = compute_entropy(torch.randn(3, 1000))

print(f"Quantization MSE: {mse:.6f}")
print("Generation Step Entropies:", entropy.tolist())
print("[SUCCESS] Quantization & Logit Extraction verified.")
